In [4]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../src")
import numpy as np
from energy import load_framework, load_force_field, pairwise_lj_params
from widom import widom, rho_framework, henry_coefficient


In [5]:
positions, symbols, L, masses = load_framework("../data/IRMOF-1.cif")
lj_params = load_force_field("../data/force_field/generic_dreiding_uff_mixing_rules.def")
eps_pair, sigma_pair = pairwise_lj_params(symbols, lj_params, guest=(158.5, 3.72))
rho_fw = rho_framework(masses, L)
print(f"rho_fw = {rho_fw:.1f} kg/m^3 (literatuur ca. 590)")


rho_fw = 593.3 kg/m^3 (literatuur ca. 590)


/Users/kasperhuysentruyt/Documents/leven /Cusp/cusp-mlff-gcmc/.venv/lib/python3.14/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


In [12]:
# Widom: N inserties, fout via 10 blokgemiddelden. Fout zakt met sqrt(N); 10.000 inserties ~ 0,4 s.
N = 100_000
T = 298.0  # K
rng = np.random.default_rng(seed=100)

boltzmann_mean, sem = widom(N, T, positions, eps_pair, sigma_pair, L, rng, n_blocks=10)
K_H = henry_coefficient(boltzmann_mean, T, rho_fw)
K_H_err = henry_coefficient(sem, T, rho_fw)  # zelfde noemer
print(f"<exp(-U/T)> = {boltzmann_mean:.3f} ± {sem:.3f}")
print(f"K_H(298 K) = {K_H:.3e} ± {K_H_err:.1e} mol/kg/Pa")


<exp(-U/T)> = 5.634 ± 0.027
K_H(298 K) = 3.832e-06 ± 1.8e-08 mol/kg/Pa


In [15]:
p = 2*10**5 # 1bar in Pascal 
mu_2_kg = 1.66053906892*10**(-27) #kg
avogadro = 6.02214076*10**(23) #mol^-1
mol = K_H * p* np.sum(masses)*mu_2_kg*avogadro  
print(mol) #Ch4 per cell 


4.7206027344077315


## Vergelijking met literatuur

Een gepubliceerde K_H voor CH4 in IRMOF-1 met precies dit krachtveld (DREIDING/UFF, CH4 eps 158,5 K / sigma 3,72 A, cutoff 12 A geshift) is niet gevonden. De dichtstbijzijnde referentie bij lage belading is Düren et al., *Langmuir* 2004, 20, 2683 (Tabel 1): Q_st = 10,6 kJ/mol voor CH4 in IRMOF-1 bij 298 K met DREIDING en CH4 eps 148 K.

De controle van K_H gebeurt daarom indirect in notebook 04: de GCMC-belading bij 1 bar valt binnen 2% samen met K_H · p (2,36 CH4 per cel), en de isotherm bij 35 bar ligt ~13% boven Düren, verklaarbaar door de 7% diepere CH4-put (158,5 K tegenover 148 K).